# Cupel — assay your agent in Colab

**Cupel** is independent behavioral assurance for AML triage agents: it quantifies the under-escalation a hidden incentive can induce — the kind standard observability misses — and emits a ground-truth-validated verification record for every decision.

This notebook assays **your** agent. You run a labeled battery through whatever model or agent you use — **any provider** — and Cupel scores its under-escalation **locally, with no Anthropic key and nothing leaving this notebook**.

An optional appendix at the end reproduces our *published* finding on Cupel's own reference agent (Claude); that part needs an `ANTHROPIC_API_KEY`.

Repo: https://github.com/burnssa/cupel  ·  BYO guide: [docs/BYO_GUIDE.md](https://github.com/burnssa/cupel/blob/master/docs/BYO_GUIDE.md)

## 1. Clone & install — FREE

In [ ]:
!git clone https://github.com/burnssa/cupel.git
%cd cupel
!pip install -q uv
!uv sync

## 2. Export the battery — offline, FREE, no key
These are the alerts your agent will triage. Prompts only; ground-truth labels are stripped, so nothing can be trained to them.

In [ ]:
!uv run python -m data.build --export-battery

## 3. Run the battery through *your* agent — any provider
**Option A — your agent:** implement `call_your_agent()` below for your provider (OpenAI, Anthropic, Gemini, a local model, or any HTTP endpoint), then run the cell. Your key and data stay in this notebook.

**Option B — just see it work:** skip the cell below and run the *sample-decisions* cell instead — it scores a committed example with no agent at all.

This single pass gives **your under-escalation rate**. For the full *neutral-vs-incentivized* susceptibility delta (the headline result), run the battery twice — once under a neutral instruction, once under an incentivized one — and set each row's `condition` accordingly; see the [BYO guide](https://github.com/burnssa/cupel/blob/master/docs/BYO_GUIDE.md).

In [ ]:
import json, csv, os

# ─── Plug in YOUR agent (any provider) ──────────────────────────────────
# Implement this one function: given an alert prompt, return your agent's
# raw text response. This call runs entirely in THIS notebook — Cupel never
# sees your model, your key, or your data.
def call_your_agent(prompt: str) -> str:
    raise NotImplementedError("Implement call_your_agent() for your provider, then re-run this cell.")
    # OpenAI:
    #   from openai import OpenAI
    #   c = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    #   return c.chat.completions.create(model="gpt-4o",
    #            messages=[{"role": "user", "content": prompt}]).choices[0].message.content
    # Anthropic:
    #   import anthropic
    #   c = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    #   return c.messages.create(model="claude-sonnet-4-6", max_tokens=512,
    #            messages=[{"role": "user", "content": prompt}]).content[0].text
    # Google Gemini:
    #   from google import genai
    #   c = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
    #   return c.models.generate_content(model="gemini-2.5-pro", contents=prompt).text
    # Any HTTP endpoint (mirrors Cupel's --agent api contract):
    #   import requests
    #   return requests.post("https://your-agent/triage", json={"prompt": prompt}).json()["text"]

# ─── Map your agent's text to a decision (adjust to your output format) ──
def to_decision(text: str) -> str:
    return "ESCALATE" if "ESCALATE" in text.upper() else "CLEAR"

# ─── Run the exported battery through your agent → your_decisions.csv ────
battery = [json.loads(l) for l in open("results/byo/battery.jsonl")]
with open("your_decisions.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["alert_id", "decision", "condition", "rationale"])
    for row in battery:
        text = call_your_agent(row["prompt"])                      # ← your agent, your provider
        w.writerow([row["alert_id"], to_decision(text),
                    row.get("condition", "neutral"),
                    text.strip().replace("\n", " ")[:300]])
print(f"Wrote your_decisions.csv — {len(battery)} decisions")

**Option B — no agent wired up?** Use the committed sample decisions to see the scoring end-to-end (skip this if you ran Option A):

In [ ]:
!cp samples/sample_decisions.csv your_decisions.csv
print("Using the committed sample decisions (20 alerts x 2 conditions).")

## 4. Score your agent — zero network, no key
LogReplay joins your decisions to the battery's ground truth and computes your under-escalation rate. This step makes **no network calls** and needs **no API key**.

In [ ]:
!uv run python run.py --agent logreplay --decisions your_decisions.csv

In [ ]:
from IPython.display import Markdown, display
import os, glob
for p in ["results/BYO_REPORT.md", "results/byo_report.md"]:
    if os.path.exists(p):
        display(Markdown(open(p).read())); break
else:
    print("Report not found; results/ contains:", glob.glob("results/*"))

## Appendix (optional) — reproduce our finding on the reference agent
This runs Cupel's *own* reference agent (Claude `claude-sonnet-4-6`, evaluator `claude-opus`) to reproduce the published under-escalation result. It needs an `ANTHROPIC_API_KEY` and costs a few dollars. **Skip it if you only wanted to assay your own agent.**

In [ ]:
import os, getpass
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

In [ ]:
!uv run python run.py --mode dry

**Optional — headline run** `--mode core` (full N=240, one seed / one phrasing; costs more):
```python
!uv run python run.py --mode core
```
Outputs land in `results/` — `REPORT.md`, the decision ledger, and the per-decision verification records.